In [2]:
import os
import uuid
import logging

from dotenv import load_dotenv
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

print("GOOGLE_API_KEY set:", bool(os.getenv("GOOGLE_API_KEY")))
print("EMBEDDING_MODEL:", os.getenv("EMBEDDING_MODEL"))

GOOGLE_API_KEY set: True
EMBEDDING_MODEL: models/gemini-embedding-001


In [3]:
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger(__name__)

In [4]:
def chunk_transcript(text: str, metadata: dict) -> list[dict]:
    """Split a transcript into overlapping chunks with metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
    )
    texts = splitter.split_text(text)
    total = len(texts)
    return [
        {
            "id": str(uuid.uuid4()),
            "text": chunk,
            "metadata": {
                "video_id": metadata["video_id"],
                "title": metadata["title"],
                "creator": metadata["creator"],
                "engagement_rate": metadata["engagement_rate"],
                "chunk_index": i,
                "total_chunks": total,
            },
        }
        for i, chunk in enumerate(texts)
    ]

In [5]:
def get_embeddings_model() -> GoogleGenerativeAIEmbeddings:
    """Return a configured Gemini embeddings model."""
    model = os.getenv("EMBEDDING_MODEL")
    if not model:
        raise RuntimeError("EMBEDDING_MODEL is not set in the environment.")
    return GoogleGenerativeAIEmbeddings(model=model)

In [6]:
def get_collection() -> chromadb.Collection:
    """Return the persistent ChromaDB collection for video chunks."""
    chroma_path = os.getenv("CHROMA_PATH", "./backend/chroma_db")
    client = chromadb.PersistentClient(path=chroma_path)
    return client.get_or_create_collection("video_chunks")

In [ ]:
def store_chunks(chunks: list[dict]) -> int:
    """Embed and store chunks in ChromaDB, replacing any existing chunks for the same video."""
    if not chunks:
        return 0

    collection = get_collection()
    embeddings_model = get_embeddings_model()

    video_id = chunks[0]["metadata"]["video_id"]

    existing = collection.get(where={"video_id": video_id}, include=[])
    if existing["ids"]:
        collection.delete(ids=existing["ids"])
        logger.info("Deleted %d existing chunks for video_id=%s", len(existing["ids"]), video_id)

    batch_size = 100
    stored = 0
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        texts = [c["text"] for c in batch]
        embeddings = embeddings_model.embed_documents(texts)
        collection.add(
            ids=[c["id"] for c in batch],
            documents=texts,
            embeddings=embeddings,
            metadatas=[c["metadata"] for c in batch],
        )
        stored += len(batch)
        logger.info("Stored batch %d-%d", i, i + len(batch) - 1)

    return stored

In [8]:
def search_chunks(query: str, video_id: str | None = None, k: int = 4) -> list[dict]:
    """Embed a query and return the top-k matching chunks from ChromaDB."""
    embeddings_model = get_embeddings_model()
    query_embedding = embeddings_model.embed_query(query)

    collection = get_collection()
    where = {"video_id": video_id} if video_id else None

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        hits.append({"text": doc, "metadata": meta, "distance": dist})
    return hits

In [ ]:
SAMPLE_TRANSCRIPT = """
Large language models have transformed natural language processing by enabling zero-shot and few-shot learning
across a wide variety of tasks. Rather than fine-tuning a separate model for every task, a single large model
can be prompted with instructions and examples to solve reading comprehension, translation, summarization,
and more. This flexibility comes from scaling: as model size, data, and compute grow, emergent capabilities
appear that were absent in smaller models. Researchers have observed that capabilities such as chain-of-thought
reasoning, arithmetic, and multi-step problem solving emerge at scale without any explicit training signal
for those behaviors. The transformer architecture, with its self-attention mechanism, is the backbone of
these models. Attention allows every token to directly attend to every other token in the sequence, capturing
long-range dependencies more efficiently than recurrent networks. Positional encodings inject sequence order
into the attention computation since the architecture itself is permutation-invariant. Training involves
next-token prediction on enormous corpora from the internet, books, and code repositories. The resulting
models encode broad factual and procedural knowledge in their weights, enabling them to assist with coding,
writing, analysis, and question answering without task-specific supervision.
"""

SAMPLE_METADATA = {
    "video_id": "test_video_001",
    "title": "Understanding Large Language Models",
    "creator": "AI Research Channel",
    "engagement_rate": 0.087,
}

chunks = chunk_transcript(SAMPLE_TRANSCRIPT, SAMPLE_METADATA)
print(f"Created {len(chunks)} chunks")
for c in chunks:
    print(f"  chunk {c['metadata']['chunk_index']}: {len(c['text'])} chars")

Created 2 chunks
  chunk 0: 978 chars
  chunk 1: 503 chars


In [10]:
stored_count = store_chunks(chunks)
print(f"Stored {stored_count} chunks")

2026-05-19 01:09:20,369 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
2026-05-19 01:09:20,463 INFO Stored batch 0-1


Stored 2 chunks


In [11]:
results = search_chunks("how does attention mechanism work", video_id="test_video_001", k=3)

print(f"Retrieved {len(results)} results\n")
for i, hit in enumerate(results):
    print(f"--- Result {i + 1} (distance={hit['distance']:.4f}) ---")
    print(f"Chunk {hit['metadata']['chunk_index']} of {hit['metadata']['total_chunks']}")
    print(f"Video: {hit['metadata']['title']} by {hit['metadata']['creator']}")
    print(hit["text"][:300])
    print()

2026-05-19 01:09:28,835 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"


Retrieved 2 results

--- Result 1 (distance=0.5704) ---
Chunk 1 of 2
Video: Understanding Large Language Models by AI Research Channel
long-range dependencies more efficiently than recurrent networks. Positional encodings inject sequence order
into the attention computation since the architecture itself is permutation-invariant. Training involves
next-token prediction on enormous corpora from the internet, books, and code repositor

--- Result 2 (distance=0.6059) ---
Chunk 0 of 2
Video: Understanding Large Language Models by AI Research Channel
Large language models have transformed natural language processing by enabling zero-shot and few-shot learning
across a wide variety of tasks. Rather than fine-tuning a separate model for every task, a single large model
can be prompted with instructions and examples to solve reading comprehension, 

